# Seoul Bike EDA

따릉이 대여이력 데이터를 Spark DataFrame으로 읽고, 최종 분석 코드 작성 전에 데이터 구조와 집계 방향을 확인한 노트북.

- 입력 데이터: HDFS processed CSV
- 최종 재현 코드: `src/pipeline/spark_analysis.py`


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("SeoulBikeEDA").getOrCreate()
spark.sparkContext.setLogLevel("WARN")


In [ ]:
input_path = "hdfs:///user/maria_dev/seoul_bike/processed/seoul_bike_2017_utf8.csv"

raw_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .option("quote", "\"") \
    .option("escape", "\"") \
    .csv(input_path)

raw_df.printSchema()
raw_df.show(5, truncate=False)


## 컬럼명 정리

원본 컬럼명은 한글이므로 Spark 코드에서 다루기 쉽도록 영어 컬럼명으로 변경한다.


In [ ]:
columns = [
    "bike_no", "rent_datetime", "rent_station_no", "rent_station_name", "rent_rack_no",
    "return_datetime", "return_station_no", "return_station_name", "return_rack_no",
    "use_min", "use_distance_m", "birth_year", "gender", "user_type",
    "rent_station_id", "return_station_id", "bike_type"
]

df = raw_df.toDF(*columns)

df = df.withColumn("rent_ts", F.to_timestamp("rent_datetime", "yyyy-MM-dd HH:mm:ss"))
df = df.withColumn("use_min_num", F.col("use_min").cast("double"))
df = df.withColumn("use_distance_m_num", F.col("use_distance_m").cast("double"))

clean_df = df.filter(F.col("rent_ts").isNotNull()) \
    .filter(F.col("use_min_num").isNotNull()) \
    .filter(F.col("use_distance_m_num").isNotNull()) \
    .filter(F.col("use_min_num") >= 0) \
    .filter(F.col("use_distance_m_num") >= 0)

clean_df.count()


## 1. 월별 이용량


In [ ]:
monthly_usage = clean_df.withColumn("month", F.date_format("rent_ts", "yyyy-MM")) \
    .groupBy("month") \
    .agg(F.count("*").alias("rental_count")) \
    .orderBy("month")

monthly_usage.show(20, truncate=False)


## 2. 시간대별 이용량


In [ ]:
hourly_usage = clean_df.withColumn("hour", F.hour("rent_ts")) \
    .groupBy("hour") \
    .agg(F.count("*").alias("rental_count")) \
    .orderBy("hour")

hourly_usage.show(24, truncate=False)


## 3. 대여소별 이용량 Top 10


In [ ]:
station_top10 = clean_df.groupBy("rent_station_no", "rent_station_name") \
    .agg(F.count("*").alias("rental_count")) \
    .orderBy(F.col("rental_count").desc()) \
    .limit(10)

station_top10.show(10, truncate=False)


## 4. 이용시간 및 이동거리 요약


In [ ]:
usage_summary = clean_df.agg(
    F.count("*").alias("row_count"),
    F.avg("use_min_num").alias("avg_use_min"),
    F.min("use_min_num").alias("min_use_min"),
    F.max("use_min_num").alias("max_use_min"),
    F.avg("use_distance_m_num").alias("avg_distance_m"),
    F.min("use_distance_m_num").alias("min_distance_m"),
    F.max("use_distance_m_num").alias("max_distance_m"),
)

usage_summary.show(truncate=False)


## 정리

이 노트북에서 확인한 집계 흐름을 `src/pipeline/spark_analysis.py`에 정리하여 `spark-submit`으로 재실행 가능하게 구성한다.
